# 9j — Forecast assembly, WIS + log-score scoring & diagnostics (two-stage cut)

Downstream half of the composable forecast, split out from `8j_preliminary_forecast.ipynb`.
Six model variants: the **four ways** (`{unweighted NegBin, weighted Hurdle-Weibull}` ×
`{mean, neighbourhood}` NGM) plus the two baselines of `inst/6_null_interaction_model.md` —
**no-interaction** (NegBin, mean, diagonal-only C*, infectivity ≡ 1) and **null** (no contact data;
C* a fixed uniform constant).

**8j fits and caches the two-stage artefacts** — Stage-1 GP chains `../dt_intermediate/8j_s1_*.jld2`
and Stage-2 pooled infection draws `8j_s2_*.jld2`; this notebook **reloads them** (no re-fit) to
assemble the pooled contact-updated forecasts (10 000 draws per origin×combo×horizon), score them
with **log-scale WIS** and the **log score** via R `scoringutils`, and draw the diagnostic figures.
**Run 8j first** — a missing artefact here only triggers a fallback re-fit.

The shared setup (`cfg`, `grid`, `raw`, `FORECAST_ORIGINS`, `wins`, `combos`) is reproduced
verbatim from 8j so the cache keys `(degree[, ngm], contacts, origin, h)` match exactly.
Outputs keep the `8j_` prefix (`res/8j_*`).

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework

using Random, Statistics
mkpath("../res")

default_plot_setting()

In [ ]:
# `constant_contacts = false` ⇒ contact degree estimated PER WEEK, temporally smoothed by a
# separable spatio-temporal GP (shared ρ_diag/ρ_gap/ρ_time, η, σ_c; scalar intercept c +
# temporal-level GP cₜ = c + σ_c·(Lt·z_c) + matrix-normal field η·Lp·z·Ltᵀ). The renewal NGM
# then varies in time through contacts as well as antibody: N(t) uses that week's C*ₜ.
# (Set true for the pooled one-C*-per-window preliminary.)
cfg  = FrameworkConfig(constant_contacts = false)
grid = cis_age_grid()

# Read the CoMix contact data ONCE and reuse it across every window (avoids re-reading/
# re-joining the full Arrow per origin×horizon). Then roll the forecast origin over the
# whole period the current datasets support ("available period"): each origin needs a
# 12-week fit/lag window back to the first inc2prev week, and contact data out to
# origin+3 for the contact-updated iterate. `available_forecast_origins` derives the range.
#
# `FIT_END` caps the roll at the last week STARTING in 2021 (⇒ last origin 2021-12-26; its h=1..4
# targets still run into Jan 2022, well inside both datasets). The data-derived bound alone is far
# too generous: CoMix's main panel stops 2022-03-02, but a stray 2022-11-16…28 block pushes it out
# to 2022-10-30, so origins from ~2022-03-06 would roll through a window with no contact data at
# all. inc2prev is a second, uncaught limit — infections/antibody end 2022-03-26 and later weeks
# are silently zero-filled rather than erroring. MUST match 8j exactly (shared cache keys).
FIT_END = Date(2021, 12, 31)     # snapped to the Sunday-start grid by week_start ⇒ 2021-12-26
raw  = load_raw_contact_inputs()
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw,
                                             origin_max = FIT_END)
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

println("contact data span  : ", extrema(skipmissing(raw.craw.date)))
println("forecast origins   : ", length(wins), " weekly, ",
        first(FORECAST_ORIGINS), " … ", last(FORECAST_ORIGINS), " (capped at ", FIT_END, ")")
let w = wins[1], wd0 = load_window_data(wins[1]; grid = grid)
    println("origin[1] fit weeks: ", w.fit_weeks[1], " … ", w.fit_weeks[end])
    println("weekly infections @ origin[1] (age): ", round.(wd0.I_mean[:, end]; digits = 0))
end

In [ ]:
# Fit config: the SIX model variants and the parallel-fit concurrency (CPU- and memory-balanced).
#   • the "four ways" 2×2 grid: {unweighted NegBin, weighted Hurdle-Weibull} × {mean, neighbourhood}
#   • no-interaction (inst/6): NegBin degree, MEAN NGM with only the DIAGONAL of C* kept, and
#     age-dependent infectivity pinned to 1 (identifiability). It reuses the NegBin Stage-1 chains
#     verbatim (`8j_s1_*` carry no ngm token) ⇒ no extra Stage-1 fits.
#   • null (inst/6): no contact data at all — C* is a fixed uniform constant, the average unweighted
#     number of contacts over the origin's 8 focal weeks, held fixed across horizons. Skips Stage 1
#     entirely; one Stage-2 fit per (origin × horizon) with the full 10 000 draws.
combos = vcat([(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                        for nb in (MeanNGM(), NeighbourhoodDegreeNGM())],
              [(NegBinAgePair(),   DiagonalMeanNGM()),      # no-interaction
               (NoContactDegree(), NullNGM())])             # null
MAX_FIT_CONCURRENCY = fit_concurrency()          # min(threads, cores−1, RAM-budget)
if Threads.nthreads() == 1
    @warn "Julia has 1 thread — pre-fit runs sequentially. Start with JULIA_NUM_THREADS>1 " *
          "(e.g. $(max(1, Sys.CPU_THREADS - 1))) for parallel fitting."
end
println("combos = ", length(combos), " | fit concurrency = ", MAX_FIT_CONCURRENCY,
        " | total fits = ", length(wins) * length(combos) * length(cfg.horizons),
        " (cached ones are skipped)")

In [ ]:
# Detailed implementation lives in 9j_viz_utils.jl — the notebook chooses inputs and calls
# the builders (assembly cache, scoring report, figures). It pulls in 8j_viz_utils.jl too.
include("9j_viz_utils.jl")

# Fixed model order + colour palette shared by the diagnostic figures below (the four ways ++ the
# two baselines from inst/6: no-interaction and null).
model_labels = [string(degree_label(dm), "|", ngm_label(nb)) for (dm, nb) in combos]
model_cols   = [:steelblue, :darkorange, :seagreen, :purple, :firebrick, :grey40]
@assert length(model_cols) >= length(model_labels)
println("REF_MODEL (relative skill) = ", REF_MODEL, "  ∈ models? ", REF_MODEL in model_labels)

# Reload the cached 8j two-stage artefacts and assemble the pooled forecast products (NO re-fit) —
# this origin×combo loop (`two_stage_forecast` → `fit_or_load_stage2` reloads the Stage-2 pooled
# 100×100 draws) is the slow step, so its outputs (`qall` quantile table, `fc_store` fans,
# `truth_store` realized infections, `crps` cross-check) are CACHED to
# `../dt_intermediate/9j_assembly_<contacts>.jld2` and reused. Set `REBUILD_ASSEMBLY = true` (or
# delete the cache) to force a fresh reload; the cache self-invalidates if origins/combos change —
# so adding the two inst/6 models forces exactly one rebuild. A missing artefact still triggers a
# fallback re-fit, so run 8j first.
REBUILD_ASSEMBLY = false
asm = assemble_or_load_forecasts(wins, combos, cfg; grid = grid, raw = raw,
                                 save_dir = "../dt_intermediate",
                                 rebuild = REBUILD_ASSEMBLY)
qall, fc_store, truth_store, crps = asm.qall, asm.fc_store, asm.truth_store, asm.crps
size(qall)

In [ ]:
scores = score_wis(qall)   # scores both natural & log scale; aggregated by horizon (inst/1e)

# Headline: log-scale WIS by model and by model×horizon + mean native-CRPS cross-check;
# writes the four res/8j_scores_*.csv frames and returns the by-model×horizon log frame.
by_mh_log = report_forecast_scores(scores, wins, crps)

In [ ]:
# log-scale WIS by horizon (one line per model), the by-model×horizon grouped bar (WIS as bar
# height — not the model index, the old bug), and WIS over the forecast period.
save_show(plot_wis_by_horizon(scores, wins, cfg), "../res/8j_wis_by_horizon.png")
save_show(plot_wis_by_model_horizon(scores, model_labels, cfg), "../res/8j_wis_by_model_horizon.png")
save_show(plot_wis_over_time(scores), "../res/8j_wis_over_time.png");

### Log score (inst/6_null_interaction_model.md)

WIS above is reported on a *log scale* (`transform_forecasts(log_shift)`) — that is a log
**transform** of the forecast, not the logarithmic **score**. `scoringutils` defines the log score
only for the **sample** forecast class, so `score_logs` runs the raw pooled draws through
`as_forecast_sample` → `score()` (KDE-based `logs_sample`), on both the natural and the log scale.

Two sanitisations happen there and are reported, not hidden: `±Inf` draws (kept deliberately by
`two_stage_forecast`) are dropped because the KDE cannot consume them, and draws are thinned to
`N_LOGS_DRAWS` per cell. The headline scale is **natural** — the log-scale variant additionally
depends on the `pmax(·,0)+1` censoring needed because individual draws can be negative.

In [ ]:
# Log score from the pooled sample fans (no re-fit — reuses `fc_store` / `truth_store`).
# One R round-trip per origin: `score()` returns one row per forecast unit, so the accumulated
# per-unit table stays small even though the sample tables handed to R are ~10^5 rows each.
N_LOGS_DRAWS = 1000   # draws retained per (age × horizon) cell for the KDE
logs = score_logs(fc_store, truth_store, wins, model_labels, cfg, grid.LAB;
                  n_sample = N_LOGS_DRAWS)

# Headline: natural-scale log score by model and by model×horizon + the sanitisation tallies;
# writes the four res/8j_logscore_*.csv frames (both scales).
by_mh_logs = report_logscore(logs, wins)

In [ ]:
# Log score by horizon (one line per model), the by-model×horizon grouped bar, the log score
# over the forecast period, and the log score RELATIVE to the no-interaction reference.
# Relative panel plots DIFFERENCES, not ratios: a log score is not sign-stable.
save_show(plot_logscore_by_horizon(logs, wins, cfg), "../res/8j_logscore_by_horizon.png")
save_show(plot_logscore_by_model_horizon(logs, model_labels, cfg),
          "../res/8j_logscore_by_model_horizon.png")
save_show(plot_logscore_over_time(logs), "../res/8j_logscore_over_time.png")
save_show(plot_rel_logscore_by_horizon(logs, model_labels, model_cols, cfg),
          "../res/9j_rel_logscore_by_horizon.png");

In [ ]:
# Relative WIS as a time series (one line per config), faceted by horizon (2×2) — each model's
# log-scale WIS ratioed to negbin|mean per horizon×date (reference on the 1.0 line, <1 = better).
save_show(plot_wis_by_horizon_over_time(scores, model_labels, model_cols, cfg),
          "../res/8j_wis_logscale_by_horizon_over_time.png");

In [ ]:
# CUMULATIVE WIS difference over time vs weighted-hweibull|neighbourhood, faceted by horizon (2×2).
# Only the two mean-NGM configs are shown (unweighted-negbin|mean, weighted-hweibull|mean): each
# model's per-origin log-scale WIS gap (wis − wis_ref) accumulated in date order. Each line's
# endpoint = the whole-period WIS difference vs the reference; < 0 ⇒ beats it cumulatively, a
# steadily-rising line loses a little every week. Dashed 0 line = the reference (omitted, ≡ 0).
save_show(plot_wis_diff_over_time(scores, model_labels, model_cols, cfg),
          "../res/9j_wis_diff_over_time.png");

In [ ]:
# HORIZON-CUMULATIVE (M-SAP) twin of the above: panels are h=1, h=1:2, h=1:3, h=1:4 — each origin
# WIS is summed over the horizon set 1:H before differencing vs weighted-hweibull|neighbourhood, then
# accumulated in date order. h=1 matches the per-horizon figure; higher panels aggregate lead-times.
save_show(plot_wis_diff_over_time_cumh(scores, model_labels, model_cols, cfg),
          "../res/9j_wis_diff_over_time_cumh.png");

In [ ]:
# WIS vs the inc2prev England Rt at the forecast TARGET week (origin+h), faceted by horizon (2×2):
# does skill degrade as transmission rises? Two models — unweighted-negbin|mean and
# weighted-hweibull|neighbourhood — one point per origin; dashed line at Rt = 1 (growth threshold).
rt_week = weekly_national_R()
save_show(plot_wis_vs_rt(scores, rt_week, cfg), "../res/9j_wis_vs_rt.png");

In [ ]:
# Per-origin WIS SCATTER: weighted-hweibull|neighbourhood (y) vs unweighted-negbin|mean (x),
# faceted by horizon (2×2). One point per forecast origin, paired on forecast_date from the
# age-aggregated log-scale WIS (by_model_dt_h). Equal square axes; dashed y=x tie line — points
# BELOW the diagonal are origins where hweibull|neighbourhood scored lower (= better).
save_show(plot_wis_scatter(scores, cfg;
                           xmodel = "unweighted-negbin|mean",
                           ymodel = "weighted-hweibull|neighbourhood"),
          "../res/9j_wis_scatter_hweibull_nbhd_vs_negbin_mean.png");

In [ ]:
# Forecast vs observed at 15 evenly-spaced origins: one observed series (fit-week history ++
# realized targets) overlaid with each model's total-infection fan (median + 90% band).
# The tile grid follows the panel count (15 ⇒ 3×5), so `n` is free to change.
save_show(plot_forecast_panels(fc_store, wins, model_labels, model_cols, cfg; grid = grid, n = 15),
          "../res/8j_forecast_vs_observed_panels.png");

In [ ]:
# Fitted transmission structure over the origins: susceptibility & infectivity as RATIOS to the
# 2-15 group from the Stage-2 pooled draws — first the 16-49 / >50 super-groups with 90% bands,
# then the same against the same 2-15 baseline at full per-age-bin resolution (median lines only;
# the super-group series are pop-weighted averages of these) — the separable-GP length-scales
# ρ_diag / ρ_gap (age-yrs) and ρ_time (weeks) from the Stage-1 chains, the per-contact
# secondary attack rate γ_SAR (one line per model; comparable across origins under un-normalised
# C*), and the ESTIMATED generation interval against its prior (added 2026-07-30).
tr = collect_transmission_structure(model_labels, FORECAST_ORIGINS; grid = grid, h = 1)
save_show(plot_ratio(tr.susc, model_labels, FORECAST_ORIGINS, "8j — susceptibility ratio to 2-15 (h=1)"),
          "../res/8j_susc_ratio_over_time.png")
save_show(plot_ratio(tr.inf, model_labels, FORECAST_ORIGINS, "8j — infectivity ratio to 2-15 (h=1)"),
          "../res/8j_infectivity_ratio_over_time.png")
save_show(plot_ratio_bins(tr.susc_bin, model_labels, FORECAST_ORIGINS,
                          "8j — susceptibility ratio to 2-15 by age group (h=1, median)"; grid = grid),
          "../res/8j_susc_ratio_bins_over_time.png")
save_show(plot_ratio_bins(tr.inf_bin, model_labels, FORECAST_ORIGINS,
                          "8j — infectivity ratio to 2-15 by age group (h=1, median)"; grid = grid),
          "../res/8j_infectivity_ratio_bins_over_time.png")
save_show(plot_lengthscales(tr.rho, model_labels, FORECAST_ORIGINS; h = 1),
          "../res/8j_lengthscale_rho_over_time.png")
save_show(plot_gamma(tr.gamma, model_labels, model_cols, FORECAST_ORIGINS; h = 1),
          "../res/8j_gamma_over_time.png")

# Generation interval — the IDENTIFIABILITY read, not just a parameter plot. w and γ_SAR are
# confounded (both scale the renewal predictor), and only the informative 20%-of-the-mean prior
# separates them. Two failure modes to look for:
#   • posterior band sitting ON the grey prior band ⇒ the data say nothing about the GI, it is
#     carrying prior only, and the extra latents are buying nothing;
#   • posterior parked on a soft-clamp with a NARROW band ⇒ clamp compression, not certainty
#     (the failure mode γ_SAR hit at log 0.02 in 2026-07-13 — suspect the clamp, not the data).
save_show(plot_gen_interval(tr.gi, model_labels, FORECAST_ORIGINS, cfg; h = 1),
          "../res/8j_generation_interval_over_time.png");

In [ ]:
# Antibody protection over time — the leaky protection factor F (one line per model, 90% band).
# F enters full_susceptibility_a(t) = susc_a·(1 + (F−1)·A_a(t)): F→0 ⇒ antibodies fully protect,
# F→1 ⇒ no protection (prior Beta(5,1) ⇒ F ∈ (0,1)). A scalar per Stage-2 draw, so this is the
# direct antibody-protection analog of the γ_SAR-over-time plot above (reuses the same `tr`, whose
# `F` field collect_transmission_structure now carries). No re-fit — reloads the pooled draws only.
save_show(plot_F(tr.F, model_labels, model_cols, FORECAST_ORIGINS; h = 1),
          "../res/9j_protection_factor_F.png");

In [ ]:
# Finest age-dependent susceptibility & infectivity WITH 90% CIs — one 3×2 figure per model
# (rows = age-bin groups split 2/2/3; columns = susceptibility, infectivity). Splitting the 7 CIS
# bins keeps ≤3 ribbons per panel readable — the CI companion to plot_ratio_bins (medians only).
# Ratio to the same pop-weighted 2-15 baseline; the per-bin lo/hi draws already live in `tr`.
for lbl in model_labels
    fn = replace(lbl, "|" => "_")
    save_show(plot_susc_inf_bins_ci(tr.susc_bin, tr.inf_bin, lbl, FORECAST_ORIGINS, grid),
              "../res/8j_susc_inf_ci_$(fn).png")
end

## Paper-style forecast diagnostics (Munday et al. 2023)

Reproduces the evaluation figures of `inst/pcbi.1011453.pdf` for our six-model forecast
grid. Reference model = `unweighted-negbin|mean-diagonal` — the real no-interaction model added
with `inst/6` (until then `unweighted-negbin|mean` stood in for it); relative
WIS and coverage use the **log scale** (the headline; robust to the neighbourhood-NGM
natural-scale blow-up).

- **Periods** (Table 2) — named UK COVID phases for aggregating skill over the timeline.
- **Fig 3** — relative WIS, bias, and age-stratified relative WIS by horizon.
- **Fig 4** — relative WIS vs horizon, faceted per pandemic period.
- **Fig 5** — 50% / 90% central-interval coverage (calibration).
- **Reproduction number** — two separate figures. *(1)* `res/9j_reproduction_number.png`: the
  "contact & transmission" R = dominant eigenvalue of the origin-week NGM over time, per
  model, with a 90% band. *(2)* `res/9j_contact_reproduction_number.png`: the "contacts only"
  **relative** contact R = `ρ(C*_t)/ρ(C*_first origin)`, the bare contact-matrix spectral radius
  anchored to the first forecast origin (=1.0), isolating contact-driven transmissibility —
  overlaid with one **model-free** line = the raw observed weekly mean-contact matrices
  (`ρ(emean_t)/ρ(emean_first origin)`, no GP / no estimate), the data-only baseline the
  fitted C* curves smooth.

Outputs are written to `res/9j_*.png` and shown inline.

In [ ]:
# Named pandemic periods (Munday 2023, Table 2) used to aggregate skill over the timeline.
# (9j_viz_utils.jl — the period map + paper-style builders — is already included above.)
period_summary(FORECAST_ORIGINS);

In [ ]:
# Fig 3 analog — (A) relative WIS & (B) bias by horizon, and (C) age-stratified relative
# WIS by horizon (one panel per CIS bin). rWIS < 1 ⇒ better than the no-interaction reference.
save_show(plot_rwis_bias_by_horizon(scores, model_labels, model_cols, cfg),
          "../res/9j_rwis_bias_by_horizon.png")
save_show(plot_rwis_by_age_horizon(scores, model_labels, model_cols, grid, cfg),
          "../res/9j_rwis_by_age_horizon.png");

In [ ]:
# Fig 4 analog — relative WIS vs horizon, faceted by pandemic period (origins mapped via
# period_of; log-scale WIS averaged over the origins in each period, then ratioed to the ref).
save_show(plot_rwis_by_period(scores, model_labels, model_cols, cfg),
          "../res/9j_rwis_by_horizon_per_period.png");

In [ ]:
# Fig 5 analog — empirical 50% / 90% central-interval coverage by horizon (calibration).
save_show(plot_interval_coverage(scores, model_labels, model_cols, cfg),
          "../res/9j_coverage_50_90.png");

In [ ]:
# Reproduction number over time — TWO SEPARATE figures at the 1-week-ahead fit (h=1):
#   • "contact & transmission" R (res/9j_reproduction_number.png) — dominant eigenvalue of the
#     frozen origin-week NGM, per pooled draw, over the inc2prev national R (England) reference
#     and the R = 1 line.
#   • "contacts only" RELATIVE R (res/9j_contact_reproduction_number.png) — dominant eigenvalue of
#     the bare contact matrix C* (no γ_SAR/susc/inf/antibody) divided by ρ(C*) at the FIRST forecast
#     origin, so each model's curve passes through 1.0 there (dashed reference). Isolates
#     contact-driven transmissibility relative to the baseline week. Overlaid with ONE model-free
#     line = the RAW observed weekly mean-contact matrices (`emp_mean`, no GP / no model estimate),
#     same first-origin anchor — the data-only baseline the fitted C* curves are smoothing.
#     The inc2prev national R (England) is overlaid on the SAME, SHARED axis, absolute and
#     unrescaled — a deliberately MIXED-UNITS plot: the steps are a ratio to the baseline contact
#     week, the red curve is an absolute R, and they are NOT a comparative quantity. Both sit near
#     1, which is the trap: the one dashed 1.0 line is the baseline week for the steps AND the
#     epidemic threshold for the red curve at once. Read the SHAPES against each other (does
#     contact-driven transmissibility turn when R turns?) — the vertical gap means nothing.
#     `national = false` drops it and restores the single-quantity figure.
# All curves (models + observed) are step functions (held constant across each week); the models
# carry a 90% band, the observed line is a black step. Reload the Stage-2 pooled files read-only
# (no re-fit); the per-origin×combo loops are slow, so each store is CACHED
# (`9j_rt_<contacts>_h<h>.jld2` / `9j_relrt_…` / `9j_obsrt_…`) and reused. Set REBUILD_RT = true
# (or delete the caches) to force a fresh compute; they self-invalidate if origins/combos change.
REBUILD_RT = false
rt_store  = reproduction_over_time_or_load(combos, model_labels, wins, cfg; grid = grid, raw = raw,
                                           h = 1, rebuild = REBUILD_RT)
crt_store = relative_contact_reproduction_over_time_or_load(combos, model_labels, wins, cfg; grid = grid,
                                                            h = 1, rebuild = REBUILD_RT)
obs_crt   = observed_contact_reproduction_over_time_or_load(wins, cfg; grid = grid, raw = raw,
                                                            h = 1, rebuild = REBUILD_RT)
p_full = plot_reproduction(rt_store, model_labels, model_cols, FORECAST_ORIGINS; h = 1)
p_rel  = plot_relative_contact_reproduction(crt_store, model_labels, model_cols, FORECAST_ORIGINS;
                                            h = 1, observed = obs_crt)
save_show(p_full, "../res/9j_reproduction_number.png")
save_show(p_rel,  "../res/9j_contact_reproduction_number.png");

## §4 Notes

- **Available period**: the forecast origin is rolled weekly over the span the current datasets
  support — bounded below by the first inc2prev week (the 12-week fit window) and above by
  `FIT_END`, which **caps the fitting period at the end of 2021** (last origin 2021-12-26; its
  1–4-week targets run into Jan 2022). The cap is deliberate: the data-derived bound (CoMix
  contact-data end, since the iterate needs contacts to origin+4) is dragged out to 2022-10-30 by
  a stray 2022-11 block even though the main panel stops 2022-03-02, and inc2prev's
  infections/antibody end 2022-03-26 and are silently zero-filled past that. Scores are written
  per model, per model×horizon, and per model×origin (`res/8j_scores_by_model*.csv`).
- **Six model variants** = the four-ways 2×2 grid ++ the two `inst/6` baselines
  (`unweighted-negbin|mean-diagonal` = no-interaction, `no-contact|null` = null). The headline
  metric is **log-scale WIS aggregated by horizon** across all origins
  (`res/8j_scores_by_model_horizon.csv`, `scale=="log"`), with over/under-prediction & dispersion
  components, bias, and 50/90% coverage. `REF_MODEL` (relative skill) is the **no-interaction**
  model.
- **Log score** (inst/6) — reported alongside WIS via the `scoringutils` **sample** class
  (`res/8j_logscore_*.csv`). Distinct from the "log-scale WIS" below: that is WIS after a log
  *transform*; this is the logarithmic *score* of the predictive density, which `scoringutils`
  cannot compute for quantile forecasts. Headline scale is **natural**; `±Inf` draws are dropped
  and draws thinned for the KDE, both counted and printed.
- **Baselines** (inst/6). *No-interaction*: only `diag(C*)` enters the NGM, so each age group's
  epidemic is self-contained and `susc_a·inf_a` is only identified as a product ⇒ `inf ≡ 1`. It is
  still allowed the **future** mean contacts (like every model here, the horizon-`h` contact window
  ends at `t₀+h`), just the diagonal part. *Null*: no contact data — a uniform constant C* whose
  level only fixes the scale of γ_SAR (forecasts are invariant to it), held fixed across horizons.
- **Two-stage cut inference** (inst/4_cut_Bayes.md): 8j fits **Stage 1** (contact-degree GP,
  `model_degree`) once per (degree × origin × horizon), then **Stage 2** (infection block,
  `model_transmission`) conditioning on each of 100 Stage-1 draws (100 Stage-2 draws each). This
  notebook reloads the pooled 10 000-draw predictive per origin×combo×horizon and scores it; a
  missing artefact triggers a fallback re-fit, so run 8j first. The raw CoMix tables are read once.
- **Per-week contact degree, temporally smoothed** (`cfg.constant_contacts = false`, the default
  here): the age-pair mean is estimated for each of the 12 window weeks by a **separable
  spatio-temporal GP** — the age-pair RBF (`ρ_diag`/`ρ_gap`) coupled across weeks by a temporal
  RBF (`ρ_time`), all shared, giving a matrix-normal field `η·Lp·z·Ltᵀ`; the overall level is a
  scalar intercept plus a decoupled temporal-level GP `cₜ = c + σ_c·(Lt·z_c)`. Each age-pair is
  thus a temporally-correlated GP (one shared `ρ_time`), **not** an independent weekly draw;
  dispersion stays per week × block (not smoothed). The renewal NGM is time-varying through
  contacts too: `N(t)` uses that week's `C*ₜ`, and the forecast uses the origin-week slice
  `C*[end]`. Set `true` for the pooled one-`C*`-per-window preliminary.
- **Reciprocity + GP smoothing of the mean** (inst/1e): the contact mean is a *symmetric*
  log-rate over the 28 unordered age pairs, `log μ_{i→j} = r_{min,max} + log(popⱼ)` (so
  `popᵢ·μ_{i→j} = popⱼ·μ_{j→i}` exactly), `r` a **separable spatio-temporal GP** over age
  midpoints (70+ → 74.5) × weeks (shared `ρ_diag`,`ρ_gap`,`ρ_time`; non-centred). The
  neighbourhood NGM stays reciprocity-balanced (size-biased C0 is not reciprocal even when μ is).
- **Secondary attack rate γ_SAR + un-normalised C*** (inst/4_cut_Bayes.md): the `-gnorm` C*/S̄
  normalisation was reverted, so C* feeds the NGM at its raw level and `gamma_sar` is the
  per-contact SAR (`N_ab = γ_SAR · susc_a(1+(F-1)A_a) · C*_ab · inf_b`); it is comparable across
  origins.
- **Group-contact duration weight** (inst/1e): `:cnt_mass=="mass"` contacts (no recorded
  duration) get `cfg.w_dur_group = 2.5/240` (fixed now, estimable later); counts unchanged.
- **Neighbourhood-degree NGM among non-zero** (inst/1c, 1d): `C0 = ⟨k²⟩/⟨k⟩ × g`,
  `g = 1/(1−P₀)` (NegBin, floored) / `g = (1−p⁰)` (hurdle-Weibull); empty per-week Weibull cells
  (⟨k⟩=0) return `C0=0`. Mean NGM (`C0 = ⟨k⟩`) unchanged.
- **Contact-updated iterate** (inst/1d): infections/antibody frozen at each origin; the contact
  matrix is re-estimated each horizon (window ending origin+h); renewal stepped one week at a
  time (mean-plugged lags).
- **WIS on a log scale** (inst/1e): `transform_forecasts(fun=log_shift, offset=1)`; both scales
  written, log is the headline. Stage 1 uses **Pathfinder** (or NUTS via 8j's `STAGE1_USE_NUTS`);
  Stage 2 uses Pathfinder.
- Remaining lean simplifications: **per-week dispersion** is not temporally smoothed (a natural
  next extension), the temporal kernel is a stationary RBF, and the transmission block and 5-day
  generation interval are reference values. Revisit before scientific interpretation.